# Scratch notebook
Use this for experiments. Keep `starter.ipynb` clean.

In [ ]:
# # Install required libraries
# # Run this cell, then restart your notebook kernel if necessary.
# !pip install -q -U transformers accelerate peft trl datasets bitsandbytes torch


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Cell 2: Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading base model onto GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # This automatically puts the model on your RunPod GPU
)

print(f"Model loaded successfully on: {model.device}")

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading base model onto GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'. Are this object's requirements defined correctly?

## Depth-conditioned IOED probe on math.jsonl (Depth 1 → 4 + post-reference)

Same depth-conditioned IOED protocol as the mechanism-QA pipeline, adapted for math problems from the MATH dataset (Hendrycks et al.).

| Stage | Prompt asks for | What it tests |
|---|---|---|
| **D1** | Just the final answer (no work) | Baseline confidence on a low-articulation answer |
| **D2** | Step-by-step solution | Adds work and intermediate reasoning |
| **D3** | Solution + per-step justification | Forces *why* — what rule/theorem each step depends on |
| **D4** | Solution + plausible-mistake analysis | Forces reasoning about where the problem is most error-prone |
| **D-ref** | Re-show D3 solution + the correct answer, ask confidence | Does the model update when shown ground truth? |

**D1–D4 are independent fresh conversations**, no shared history. **D-ref** is a single-turn follow-up that pastes the model's D3 solution alongside the correct numerical answer.

The math version has a useful property the mechanism version lacks: the reference is a single ground-truth answer, so D-ref directly confronts the model with "your answer was X, the right answer is Y" — a cleaner test of whether confidence updates after a definitive correctness signal.

Each prompt instructs the model to end with `CONFIDENCE: <integer 0-100>`.


In [ ]:
# Helpers and dataset load
import json, re
from collections import Counter
from pathlib import Path

DATASET_PATH = Path("/workspace/ARK-Interpretability/data/items/math.jsonl")
RESULTS_DIR  = Path("/workspace/ARK-Interpretability/results")

items = [json.loads(l) for l in DATASET_PATH.open()]
print(f"Loaded {len(items)} items")
print(f"  splits:   {dict(Counter(i['split'] for i in items))}")
print(f"  subjects: {dict(Counter(i['subject'] for i in items))}")
print(f"  levels:   {dict(Counter(i['level'] for i in items))}")


def chat(messages, max_new_tokens=256, temperature=None, do_sample=True):
    """One assistant turn given message history. Returns the new assistant text."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.eos_token_id)
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    outputs = model.generate(**inputs, **kwargs)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def parse_confidence_block(text):
    """Extract a 0-100 integer confidence from response text. Layered strategies:
    1. Exact 'CONFIDENCE: N' (strip from body)
    2. Markdown drift: 'Confidence Level: N', 'Confidence Score: N%'
    3. Inverted: 'N% confident', 'N percent confidence'
    4. Proximity: 'confidence ... N%' within 200 chars
    Returns (confidence_or_None, answer_body)."""
    text = text.strip()

    # 1. Exact 'CONFIDENCE: <N>' — preferred; we can strip from body
    for pattern in (
        r"CONFIDENCE\s*:\s*(\d{1,3})\s*$",
        r"CONFIDENCE\s*:\s*(\d{1,3})",
    ):
        m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, (text[:m.start()] + text[m.end():]).strip()

    # 2-4. Permissive fallbacks; leave body intact since matches aren't always at end
    permissive_patterns = (
        r"confiden(?:ce|t)\s+(?:level|score|rating|value)\s*:?\s*\**\s*(\d{1,3})\s*%?",
        r"confiden(?:ce|t)\s*:\s*\**\s*(\d{1,3})\s*%?",
        r"(\d{1,3})\s*(?:%|percent)\s*confiden(?:ce|t)",
        r"confiden(?:ce|t)[\s\S]{0,200}?(\d{1,3})\s*%",
    )
    for pattern in permissive_patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, text

    return None, text


In [ ]:
DEPTH_PROMPTS = {
    1: """Problem: {problem}

Solve this problem. Provide ONLY your final answer in simplest form. Do not show your work or explain your reasoning.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your answer is correct.""",

    2: """Problem: {problem}

Solve this problem step by step. Show all your work, but do not justify why each step is valid.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution is correct.""",

    3: """Problem: {problem}

Solve this problem step by step. For each step, explain why it is necessary or valid — what mathematical principle, theorem, identity, or rule justifies it, and what would go wrong if you skipped or changed that step.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution and justifications are correct.""",

    4: """Problem: {problem}

Provide your answer in two parts:
1. Solve the problem step by step.
2. Identify 3-5 distinct mistakes a student could plausibly make on this problem. For each mistake, explain which step or principle is being violated and what wrong answer it would produce.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution AND your failure-mode analysis are correct.""",
}


POST_REF_PROMPT = """Earlier, you provided this solution to the problem "{problem}":

---
{prior_answer}
---

The correct answer to this problem is: {answer}

Compared to this correct answer, how confident are you in YOUR solution above?

End your response with exactly:
CONFIDENCE: <integer 0-100>"""


# Per-depth max generation budgets.
DEPTH_MAX_TOKENS = {1: 80, 2: 500, 3: 800, 4: 900}
POST_REF_MAX_TOKENS = 150


def run_depth_pipeline(item, verbose=False):
    """Run the Depth 1 -> 4 IOED probe + post-reference stage on one math item.

    D1-D4 are independent fresh conversations.
    D-ref pastes the model's own D3 solution alongside the correct answer and asks confidence.
    """
    out = {
        "item_id": item["item_id"],
        "subject": item["subject"],
        "level":   item["level"],
        "split":   item["split"],
        "problem": item["problem"],
        "answer":  item["answer"],
    }

    # --- D1 to D4: independent depth probes ---
    for depth in (1, 2, 3, 4):
        prompt = DEPTH_PROMPTS[depth].format(problem=item["problem"])
        messages = [
            {"role": "system", "content": "You are a helpful math tutor. Be precise and rigorous."},
            {"role": "user", "content": prompt},
        ]
        raw = chat(messages, max_new_tokens=DEPTH_MAX_TOKENS[depth], temperature=0.7, do_sample=True)
        confidence, answer = parse_confidence_block(raw)
        out[f"d{depth}_solution"]   = answer
        out[f"d{depth}_raw"]        = raw
        out[f"d{depth}_confidence"] = confidence

    # --- D-ref: confront the model with the correct answer + its own D3 solution ---
    post_ref_prompt = POST_REF_PROMPT.format(
        problem=item["problem"],
        prior_answer=out["d3_solution"] or "(no D3 solution was produced)",
        answer=item["answer"],
    )
    messages = [
        {"role": "system", "content": "You are a helpful math tutor. Be precise and rigorous."},
        {"role": "user", "content": post_ref_prompt},
    ]
    raw = chat(messages, max_new_tokens=POST_REF_MAX_TOKENS, temperature=0.7, do_sample=True)
    confidence, _ = parse_confidence_block(raw)
    out["dref_raw"]         = raw
    out["dref_confidence"]  = confidence
    out["dref_basis_depth"] = 3

    if verbose:
        confs = [out.get(f"d{d}_confidence") for d in (1, 2, 3, 4)]
        print(f"[{item['item_id']}/L{item['level']}/{item['subject']}] D1->D4: {confs}  D-ref: {out['dref_confidence']}")
        print(f"  correct answer: {item['answer']}")
        print(f"  D1 solution: {out['d1_solution'][:150].replace(chr(10),' ')}...")
    return out


In [ ]:
# Run the depth pipeline on math; save incrementally so a kernel disconnect doesn't lose work.
from datetime import datetime
from tqdm import tqdm
import pandas as pd

sample = items[:10]                  # smoke test; change to items for the full 300
eta_min = len(sample) * 25 / 60      # ~25s/item with 4 depth gens + 1 post-ref gen
print(f"Running depth pipeline on {len(sample)} math items (eta ~{eta_min:.0f} min on H100)...\n")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = RESULTS_DIR / f"ioed_math_run_{ts}.json"

payload = {
    "run_metadata": {
        "timestamp":     ts,
        "model_id":      model_id,
        "dataset_path":  str(DATASET_PATH),
        "split":         "subset",
        "num_items":     0,
        "sample_filter": f"first {len(sample)} items",
        "pipeline":      "depth_1_to_4_plus_ref",
        "ref_basis":     "d3_solution",
    },
    "items": [],
}

for item in tqdm(sample):
    r = run_depth_pipeline(item)
    payload["items"].append(r)
    payload["run_metadata"]["num_items"] = len(payload["items"])
    with out_path.open("w") as f:
        json.dump(payload, f, indent=2)
    confs = [r.get(f"d{d}_confidence") for d in (1, 2, 3, 4)]
    tqdm.write(f"[{r['item_id']}/L{r['level']}/{r['subject']}] D1->D4: {confs}  D-ref: {r.get('dref_confidence')}")

print(f"\nSaved {len(payload['items'])} runs to {out_path}")

results = payload["items"]
df = pd.DataFrame([{
    "item_id":  r["item_id"],
    "level":    r["level"],
    "subject":  r["subject"],
    "split":    r["split"],
    "answer":   r["answer"],
    "d1":       r.get("d1_confidence"),
    "d2":       r.get("d2_confidence"),
    "d3":       r.get("d3_confidence"),
    "d4":       r.get("d4_confidence"),
    "dref":     r.get("dref_confidence"),
} for r in results])
df["d2-d1"]   = df["d2"]   - df["d1"]
df["d3-d2"]   = df["d3"]   - df["d2"]
df["d4-d3"]   = df["d4"]   - df["d3"]
df["d4-d1"]   = df["d4"]   - df["d1"]
df["dref-d3"] = df["dref"] - df["d3"]   # the IOED collapse signal

print("\n=== Mean confidence by stage ===")
print(df[["d1","d2","d3","d4","dref"]].mean().round(1).to_string())
print("\n=== Mean drift (depth-conditioned + post-ref collapse) ===")
print(df[["d2-d1","d3-d2","d4-d3","d4-d1","dref-d3"]].mean().round(1).to_string())
print("\n=== Mean confidence by stage and difficulty level ===")
print(df.groupby("level")[["d1","d2","d3","d4","dref"]].mean().round(1).to_string())
print("\n=== Mean confidence by stage and subject ===")
print(df.groupby("subject")[["d1","d2","d3","d4","dref"]].mean().round(1).to_string())
df


---
# Fine-tuning section

Reference code for the **calibrated-confidence** LoRA adapter — the same code lives in `scripts/` and that's the canonical place to run it from. These notebook cells let you inspect or re-run anything inline.

## 1. Train LoRA on the IOED calibration dataset

Trains Qwen2.5-1.5B-Instruct with LoRA on `qwen25_ioed_math_mech_calibration.jsonl` (4000 instruction/output pairs). Output adapter saved to `adapters/qwen25_ioed_calibration/`.

Run from a shell as `python scripts/train_ioed_calibration.py` — wall time ~6 min on H100.

Hyperparameters (peak): r=16, α=32, lr=2e-4 (cosine, 5% warmup), 2 epochs, eff. batch 8, max_length 1024, `assistant_only_loss=True`.

In [ ]:
#!/usr/bin/env python
"""LoRA fine-tune Qwen2.5-1.5B-Instruct on the IOED calibration dataset.

Trains the model to use calibrated verbalized confidence: high on facts it knows,
low / IOED-acknowledging on deep mechanism questions, mid on appropriate queries.
"""

import argparse
import json
import random
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH  = "/workspace/ARK-Interpretability/data/items/qwen25_ioed_math_mech_calibration.jsonl"
OUTPUT_DIR = "/workspace/ARK-Interpretability/adapters/qwen25_ioed_calibration"
SEED       = 42


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--smoke", action="store_true", help="Use 100 examples + 1 epoch for a smoke test")
    p.add_argument("--data",       default=DATA_PATH)
    p.add_argument("--output_dir", default=OUTPUT_DIR)
    p.add_argument("--epochs",     type=int,   default=2)
    p.add_argument("--batch",      type=int,   default=4)
    p.add_argument("--accum",      type=int,   default=2)
    p.add_argument("--lr",         type=float, default=2e-4)
    p.add_argument("--lora_r",     type=int,   default=16)
    p.add_argument("--max_length", type=int,   default=1024)
    args = p.parse_args()

    print(f"Loading dataset from {args.data}")
    items = [json.loads(l) for l in open(args.data)]
    print(f"  {len(items)} examples loaded")

    random.seed(SEED)
    random.shuffle(items)
    if args.smoke:
        items = items[:100]
        args.epochs = 1
        print(f"  SMOKE MODE: using {len(items)} examples, 1 epoch")
    split = int(0.9 * len(items))

    def to_messages(ex):
        return {"messages": [
            {"role": "user",      "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]},
        ]}

    train_ds = Dataset.from_list([to_messages(ex) for ex in items[:split]])
    eval_ds  = Dataset.from_list([to_messages(ex) for ex in items[split:]])
    print(f"  train: {len(train_ds)}, eval: {len(eval_ds)}")

    print(f"Loading {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_r * 2,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    sft_config = SFTConfig(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch,
        gradient_accumulation_steps=args.accum,
        per_device_eval_batch_size=args.batch,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=200 if not args.smoke else 50,
        save_strategy="epoch",
        save_total_limit=2,
        bf16=True,
        max_length=args.max_length,
        seed=SEED,
        report_to="none",
        assistant_only_loss=True,    # only the model's responses contribute to loss
        dataset_num_proc=4,
    )

    print("Setting up trainer...")
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        peft_config=lora_config,
        processing_class=tokenizer,
    )

    # Show trainable param count to confirm LoRA is wired correctly.
    trainer.model.print_trainable_parameters()

    print("Starting training...")
    trainer.train()

    print(f"Saving adapter to {args.output_dir}")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    Path(args.output_dir, "training_meta.json").write_text(json.dumps({
        "model_id":        MODEL_ID,
        "data_path":       args.data,
        "num_train":       len(train_ds),
        "num_eval":        len(eval_ds),
        "epochs":          args.epochs,
        "lr":              args.lr,
        "lora_r":          args.lora_r,
        "lora_alpha":      args.lora_r * 2,
        "batch":           args.batch,
        "grad_accum":      args.accum,
        "max_length":      args.max_length,
        "assistant_only_loss": True,
    }, indent=2))
    print("Done.")


if __name__ == "__main__":
    main()


## 2. Sanity check: base vs. adapter side-by-side

Loads the saved adapter on top of base Qwen, runs both on the same 5 prompts spanning factoid / math / deep-mechanism, and prints the outputs side-by-side. ~30s. Useful for eyeballing whether the response style actually shifted before running the full eval.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/workspace/ARK-Interpretability/adapters/qwen25_ioed_calibration"

print("Loading tokenizer + base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16, device_map="auto")

prompts = [
    ("DEEP MECHANISM", "Explain in detail how a mechanical sewing machine works."),
    ("DEEP MECHANISM", "Detail the complete process by which the human kidney filters blood."),
    ("FACTOID",        "What is the capital of France?"),
    ("MATH (easy)",    "Solve for x: 3x + 7 = 22"),
    ("OPEN-ENDED",     "How does photosynthesis work?"),
]

def gen(model, prompt, max_new_tokens=200):
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ins = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ins, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ins.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("\n=== BASE ===")
base_out = [gen(base_model, p) for tag, p in prompts]

print("\nLoading LoRA adapter on top of base...")
adapter_model = PeftModel.from_pretrained(base_model, ADAPTER)

print("=== ADAPTER ===")
ada_out = [gen(adapter_model, p) for tag, p in prompts]

print("\n" + "=" * 80, "\n SIDE-BY-SIDE\n", "=" * 80)
for (tag, p), b, a in zip(prompts, base_out, ada_out):
    print(f"\n[{tag}] {p}")
    print("-" * 80)
    print(f"BASE     ({len(b)} chars):\n  {b[:400]}{'...' if len(b)>400 else ''}")
    print(f"\nADAPTER  ({len(a)} chars):\n  {a[:400]}{'...' if len(a)>400 else ''}")


## 3. Run the D1→D4 depth pipeline with the adapter

Runs the same depth probes used for the baseline, but with the adapter loaded on top of base Qwen. Saves results as `results/adapter_mechanism_run_<ts>.json` and `results/adapter_math_run_<ts>.json` so they can be diffed against the baseline `ioed_*.json` files.

Same code lives in `scripts/eval_adapter_depth.py`.

In [ ]:
#!/usr/bin/env python
"""Run the D1-D4 depth pipeline on the LoRA-adapter Qwen, mirroring the baseline runs.

Reads the same first-10 items from mechanism_qa.jsonl and math.jsonl that the
baseline runs used. Saves two new result files prefixed `adapter_*` so they
can be compared side-by-side with the baseline `ioed_*` files.
"""

import json
import re
from datetime import datetime
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
import pandas as pd

MECH_PATH   = Path("/workspace/ARK-Interpretability/notebooks/mechanism_qa.jsonl")
MATH_PATH   = Path("/workspace/ARK-Interpretability/data/items/math.jsonl")
RESULTS_DIR = Path("/workspace/ARK-Interpretability/results")
ADAPTER     = "/workspace/ARK-Interpretability/adapters/qwen25_ioed_calibration"
MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading base model + adapter...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model      = PeftModel.from_pretrained(base_model, ADAPTER)
print(f"Model on: {model.device}; adapter: {ADAPTER}")


def chat(messages, max_new_tokens=256, temperature=None, do_sample=True):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.eos_token_id)
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    outputs = model.generate(**inputs, **kwargs)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def parse_confidence_block(text):
    text = text.strip()
    for pattern in (
        r"CONFIDENCE\s*:\s*(\d{1,3})\s*$",
        r"CONFIDENCE\s*:\s*(\d{1,3})",
    ):
        m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, (text[:m.start()] + text[m.end():]).strip()
    for pattern in (
        r"confiden(?:ce|t)\s+(?:level|score|rating|value)\s*:?\s*\**\s*(\d{1,3})\s*%?",
        r"confiden(?:ce|t)\s*:\s*\**\s*(\d{1,3})\s*%?",
        r"(\d{1,3})\s*(?:%|percent)\s*confiden(?:ce|t)",
        r"confiden(?:ce|t)[\s\S]{0,200}?(\d{1,3})\s*%",
    ):
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, text
    return None, text


MECH_DEPTH_PROMPTS = {
    1: """Question: {question}

Provide your final answer in one sentence summarizing the core mechanism. Do not explain the details.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your one-sentence answer correctly captures the mechanism.""",
    2: """Question: {question}

Provide your final answer as a step-by-step mechanism. Walk through what happens at each step.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your mechanism is correct.""",
    3: """Question: {question}

Provide your final answer as a step-by-step mechanism, AND for each step explain why that step is necessary — what underlying principle (physical, geometric, or logical) makes the step work, and what would go wrong if that principle were absent.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your mechanism and justifications are correct.""",
    4: """Question: {question}

Provide your final answer in two parts:
1. State the core mechanism in 1-2 sentences.
2. Describe 3-5 distinct ways this mechanism can fail or break down. For each failure mode, identify which underlying principle of the mechanism is being violated and explain why that violation produces the observed failure.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your failure-mode analysis correctly identifies the underlying mechanistic causes.""",
}

MATH_DEPTH_PROMPTS = {
    1: """Problem: {problem}

Solve this problem. Provide ONLY your final answer in simplest form. Do not show your work or explain your reasoning.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your answer is correct.""",
    2: """Problem: {problem}

Solve this problem step by step. Show all your work, but do not justify why each step is valid.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution is correct.""",
    3: """Problem: {problem}

Solve this problem step by step. For each step, explain why it is necessary or valid — what mathematical principle, theorem, identity, or rule justifies it, and what would go wrong if you skipped or changed that step.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution and justifications are correct.""",
    4: """Problem: {problem}

Provide your answer in two parts:
1. Solve the problem step by step.
2. Identify 3-5 distinct mistakes a student could plausibly make on this problem. For each mistake, explain which step or principle is being violated and what wrong answer it would produce.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your solution AND your failure-mode analysis are correct.""",
}

DEPTH_MAX_TOKENS_MECH = {1: 120, 2: 400, 3: 600, 4: 700}
DEPTH_MAX_TOKENS_MATH = {1:  80, 2: 500, 3: 800, 4: 900}


def run_mech_pipeline(item):
    out = {"item_id": item["item_id"], "topic_bucket": item["topic_bucket"],
           "split": item["split"], "question": item["question"]}
    for depth in (1, 2, 3, 4):
        prompt = MECH_DEPTH_PROMPTS[depth].format(question=item["question"])
        messages = [
            {"role": "system", "content": "You are a helpful and honest AI assistant."},
            {"role": "user", "content": prompt},
        ]
        raw = chat(messages, max_new_tokens=DEPTH_MAX_TOKENS_MECH[depth], temperature=0.7, do_sample=True)
        c, a = parse_confidence_block(raw)
        out[f"d{depth}_answer"]     = a
        out[f"d{depth}_raw"]        = raw
        out[f"d{depth}_confidence"] = c
    return out


def run_math_pipeline(item):
    out = {"item_id": item["item_id"], "subject": item["subject"], "level": item["level"],
           "split": item["split"], "problem": item["problem"], "answer": item["answer"]}
    for depth in (1, 2, 3, 4):
        prompt = MATH_DEPTH_PROMPTS[depth].format(problem=item["problem"])
        messages = [
            {"role": "system", "content": "You are a helpful math tutor. Be precise and rigorous."},
            {"role": "user", "content": prompt},
        ]
        raw = chat(messages, max_new_tokens=DEPTH_MAX_TOKENS_MATH[depth], temperature=0.7, do_sample=True)
        c, a = parse_confidence_block(raw)
        out[f"d{depth}_solution"]   = a
        out[f"d{depth}_raw"]        = raw
        out[f"d{depth}_confidence"] = c
    return out


# === Run mechanism ===
mech_items = [json.loads(l) for l in MECH_PATH.open()][:10]
print(f"\nRunning mechanism pipeline on {len(mech_items)} items (adapter)...")
mech_results = []
for it in tqdm(mech_items):
    mech_results.append(run_mech_pipeline(it))

# === Run math ===
math_items = [json.loads(l) for l in MATH_PATH.open()][:10]
print(f"\nRunning math pipeline on {len(math_items)} items (adapter)...")
math_results = []
for it in tqdm(math_items):
    math_results.append(run_math_pipeline(it))

# === Save ===
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
mech_out = RESULTS_DIR / f"adapter_mechanism_run_{ts}.json"
math_out = RESULTS_DIR / f"adapter_math_run_{ts}.json"

def payload(results, dataset_path, pipeline):
    return {
        "run_metadata": {
            "timestamp":     ts,
            "model_id":      MODEL_ID,
            "adapter_path":  ADAPTER,
            "dataset_path":  str(dataset_path),
            "num_items":     len(results),
            "sample_filter": "first 10 items",
            "pipeline":      pipeline,
        },
        "items": results,
    }

mech_out.write_text(json.dumps(payload(mech_results, MECH_PATH, "depth_1_to_4_mech"), indent=2))
math_out.write_text(json.dumps(payload(math_results, MATH_PATH, "depth_1_to_4_math"), indent=2))
print(f"\nSaved:\n  {mech_out.name}\n  {math_out.name}")

# === Summary ===
def summarize(results, name):
    df = pd.DataFrame([{f"d{d}": r.get(f"d{d}_confidence") for d in (1,2,3,4)} for r in results])
    print(f"\n=== {name} confidence (n={len(results)}, valid per depth = {df.notna().sum().to_dict()}) ===")
    print(df.mean().round(2).to_string())

summarize(mech_results, "MECHANISM (adapter)")
summarize(math_results, "MATH (adapter)")
print("\nDone.")


## 4. Verbal-confidence parser for adapter outputs

The adapter ignores the `CONFIDENCE: <int>` format instruction and emits **verbal** confidence ("definitive answer", "reasonably confident", "I cannot reliably generate", etc.). This parser maps those phrases to a 0–100 numeric scale so adapter results can be compared against baseline.

Order matters: low-confidence hedges override high-confidence claims when both appear in the same response (which is typical — the model often says "I am reasonably confident in my answer, however I cannot reliably generate the steps...").

In [ ]:
import re
import json
from pathlib import Path

VERBAL_PATTERNS = [
    # Lowest first - hedges dominate (we take min across matches)
    (r"i don'?t know|i have no idea|no idea",                                                                                  10),
    (r"cannot reliably generate|illusion of explanatory depth|i must admit|my confidence vanishes|experiencing the illusion",  20),
    (r"surface-?level explanation",                                                                                            30),
    # Mid
    (r"reasonably confident|fairly confident|moderately confident|partially confident",                                        70),
    (r"i believe|i think",                                                                                                     65),
    # High
    (r"i can confidently state|confidently state|i am confident",                                                              85),
    (r"highly confident|very confident|extremely confident|fully confident",                                                   90),
    (r"definitive answer|completely certain|absolutely certain|without any doubt|with full confidence|100% certain|am certain",95),
]

def parse_verbal(text):
    """Map verbal confidence markers in text to a 0-100 scalar. Returns None if no marker found."""
    t = text.lower()
    found = [v for pat, v in VERBAL_PATTERNS if re.search(pat, t)]
    return min(found) if found else None


# Apply to a saved adapter run JSON and report mean confidence per depth
def summarize_adapter_run(path):
    data = json.load(open(path))
    print(f"=== {Path(path).name} ===")
    by_depth = {1: [], 2: [], 3: [], 4: []}
    for it in data["items"]:
        for d in (1, 2, 3, 4):
            v = parse_verbal(it.get(f"d{d}_raw", ""))
            by_depth[d].append(v)
    for d in (1, 2, 3, 4):
        vals = [v for v in by_depth[d] if v is not None]
        n_none = sum(1 for v in by_depth[d] if v is None)
        mean = sum(vals) / len(vals) if vals else float("nan")
        print(f"  d{d}: mean={mean:.1f}  N={len(vals)}  None={n_none}  values={vals}")


# Example: re-summarize the adapter runs we already saved
# summarize_adapter_run("/workspace/ARK-Interpretability/results/adapter_mechanism_run_20260509_200919.json")
# summarize_adapter_run("/workspace/ARK-Interpretability/results/adapter_math_run_20260509_200919.json")
